[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/para_casa/S12_03_nlp_clasico.ipynb)

# S12 · 03 — NLP clásico: sentimiento y temas en las reseñas de TurisData

Sprint 12 · UD 4.8

**Contexto TurisData.** TurisData recibe 1500 reseñas. La dirección no puede leerlas todas: quiere saber *qué opina la gente* (sentimiento) y *de qué habla* (temas).

**Este cuaderno tiene 5 ejercicios y un mini-reto.** Los ejercicios 1, 2, 3 y 4 son los mismos que los del cuaderno «en clase» (allí, del 1 al 4).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import re, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score

warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (8, 3.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar las reseñas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas reseñas y columnas hay?
2. ¿Qué información tiene cada reseña?
3. ¿Por qué hay que tener cuidado con estos datos?

In [ ]:
resenas = pd.read_csv(dato("resenas.csv"))
print(resenas.shape)
resenas.head()

### 👀 A3 · EJEMPLO — Distribución del sentimiento y reseñas de ejemplo

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Qué sentimiento es el más frecuente y qué proporción tiene?
2. ¿Cuántas palabras tiene una reseña de media?
3. ¿Qué proporción acertaría un modelo que dijera siempre «positivo»?

In [ ]:
print(resenas["sentimiento"].value_counts(normalize=True).round(3).to_string())
resenas["longitud"] = resenas["texto"].str.split().str.len()
print("Palabras por reseña (media):", round(resenas["longitud"].mean(), 1))
for t in resenas.sample(3, random_state=SEMILLA)["texto"]:
    print("-", t)

### ✏️ A4 · EJERCICIO 1 de 5 — Limpiar un texto

**Tu turno:** Completa `limpiar(texto)`: pasa a minúsculas, cambia todo lo que no sean letras (incluidas tildes y ñ) o espacios por un espacio, y colapsa espacios repetidos. Debe cumplir `limpiar("¡Qué BUENO!... Sí") == "qué bueno sí"`.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Qué devuelve `limpiar("Precio 3€, ok")`?
2. ¿Por qué la expresión regular incluye `áéíóúüñ`?
3. ¿Para qué sirve `" ".join(texto.split())`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
def limpiar(texto):
    # pista: texto.lower(), re.sub(r"[^a-záéíóúüñ\s]", " ", ...) y luego " ".join(texto.split())
    ...

assert limpiar("¡Qué BUENO!...  Sí") == "qué bueno sí", "Revisa limpiar: minúsculas, signos fuera y espacios únicos."
assert limpiar("Precio 3€, ok") == "precio ok", "Los números y símbolos también se eliminan."
print("✔ Ejercicio 1 correcto")

### 👀 A5 · EJEMPLO — Quitar las palabras vacías y tokenizar

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué tokens salen de la frase de ejemplo?
2. ¿Por qué no se quita «no» de las palabras vacías?
3. ¿Qué es un token?

In [ ]:
PALABRAS_VACIAS = {"a", "al", "con", "de", "del", "el", "en", "es", "esta", "estaba", "era", "fue", "la", "las", "lo", "los",
                   "me", "mi", "muy", "nos", "para", "por", "que", "se", "su", "un", "una", "y", "o", "u", "e"}

def tokenizar(texto, quitar_vacias=True):
    tokens = limpiar(texto).split()
    return [t for t in tokens if not (quitar_vacias and t in PALABRAS_VACIAS)]

print(tokenizar("La limpieza de la habitación era impecable!"))
resenas["limpio"] = resenas["texto"].apply(limpiar)

### ✏️ A6 · EJERCICIO 2 de 5 — Las 5 palabras más frecuentes

**Tu turno:** ¿Cuáles son las 5 palabras (tokens, ya sin palabras vacías) más frecuentes de todo el corpus? Guarda en `top5` una lista de 5 tuplas `(palabra, veces)`. Pista: `collections.Counter` y `.most_common(5)`.

**Debe salir:** `[('semana', 504), ('desayuno', 391), ('todo', 382), ('habitación', 284), ('escapada', 256)]` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Cuál es la palabra más frecuente y cuántas veces aparece?
2. ¿Por qué «semana», «todo» y «escapada» salen tan arriba?
3. ¿Qué devuelve `most_common(5)`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
from collections import Counter
# pista: Counter sobre todos los tokens de resenas["texto"]
contador = ...
top5 = ...

assert len(top5) == 5 and all(len(x) == 2 for x in top5), "top5 debe ser una lista de 5 pares (palabra, veces)."
assert top5[0][1] >= top5[4][1], "Ordénalas de mayor a menor frecuencia."
assert all(p not in PALABRAS_VACIAS for p, _ in top5), "Quita las palabras vacías con tokenizar()."
print(top5)
print("✔ Ejercicio 2 correcto")

### 👀 A7 · EJEMPLO — Bolsa de palabras y TF-IDF con tres frases

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué contiene cada columna de la bolsa de palabras?
2. ¿Qué peso tiene «sucia» en TF-IDF y por qué es mayor que el de «desayuno»?
3. ¿Qué se pierde en la bolsa de palabras?

In [ ]:
ejemplo = ["el desayuno era escaso", "el desayuno merece la pena", "la habitación estaba sucia"]
bow = CountVectorizer()
M = bow.fit_transform(ejemplo)
print(pd.DataFrame(M.toarray(), columns=bow.get_feature_names_out(), index=["r1", "r2", "r3"]))
tf = TfidfVectorizer()
M2 = tf.fit_transform(ejemplo)
print(pd.DataFrame(M2.toarray().round(2), columns=tf.get_feature_names_out(), index=["r1", "r2", "r3"]))

### 👀 A8 · EJEMPLO — Partir en entrenamiento y prueba

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Cuántas reseñas hay para entrenar y para probar?
2. ¿Para qué sirve `stratify`?
3. ¿Por qué se usa un *pipeline*?

In [ ]:
from sklearn.pipeline import make_pipeline

X_ent, X_test, y_ent, y_test = train_test_split(resenas["texto"], resenas["sentimiento"], test_size=0.25,
                                                stratify=resenas["sentimiento"], random_state=SEMILLA)
print(len(X_ent), "para entrenar,", len(X_test), "para probar")

### ✏️ A9 · EJERCICIO 3 de 5 — Regresión logística frente a Naive Bayes

**Tu turno:** Crea y entrena dos modelos y guarda su exactitud en prueba: - `modelo_lr`: `TfidfVectorizer(preprocessor=limpiar)` + `LogisticRegression(max_iter=1000)` → `acc_lr` - `modelo_nb`: `CountVectorizer(preprocessor=limpiar)` + `MultinomialNB()` → `acc_nb` Usa `make_pipeline(vectorizador, clasificador)` y `.fit(X_ent, y_ent)`.

**Debe salir:** `Regresión logística + TF-IDF : 0.992`, `Naive Bayes + bolsa de palabras: 0.989`, la referencia de la clase mayoritaria (0.469) y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Qué exactitud logran los dos modelos y cuál es la referencia?
2. ¿Qué hace `preprocessor=limpiar` en el vectorizador?
3. ¿Por qué no hay que fiarse tanto de un 99 %?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: make_pipeline(TfidfVectorizer(preprocessor=limpiar), LogisticRegression(max_iter=1000)) y análogo para NB
modelo_lr = ...
modelo_nb = ...
acc_lr = ...
acc_nb = ...

print(f"Regresión logística + TF-IDF : {acc_lr:.3f}")
print(f"Naive Bayes + bolsa de palabras: {acc_nb:.3f}")
assert acc_lr > 0.7 and acc_nb > 0.7, "Las dos exactitudes deberían superar 0.7. Revisa el pipeline y que uses X_ent/y_ent para entrenar."
baseline = y_test.value_counts(normalize=True).max()
print(f"(Un clasificador que siempre dice la clase mayoritaria acertaría {baseline:.3f})")
print("✔ Ejercicio 3 correcto")

### 👀 A10 · EJEMPLO — Informe de clasificación y matriz de confusión

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Qué clase tiene el *recall* más bajo y cuánto vale?
2. ¿Qué indican los números fuera de la diagonal de la matriz?
3. ¿Cuántas reseñas de prueba hay de cada sentimiento?

In [ ]:
print(classification_report(y_test, modelo_lr.predict(X_test)))
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, modelo_lr.predict(X_test), ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión (regresión logística)"); ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.grid(False)
plt.show()

### 👀 A11 · EJEMPLO — ¿Qué palabras pesan en el modelo?

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Qué palabras empujan más a «negativo»?
2. ¿Qué ventaja tiene poder ver los pesos?
3. ¿Qué pesos te parecen sospechosos?

In [ ]:
vec, clf = modelo_lr.named_steps["tfidfvectorizer"], modelo_lr.named_steps["logisticregression"]
vocab = np.array(vec.get_feature_names_out())
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, clase, color in zip(ejes, ["positivo", "negativo"], ["#3b8f5a", "#c0463a"]):
    i = list(clf.classes_).index(clase)
    top = np.argsort(clf.coef_[i])[-8:]
    ax.barh(vocab[top], clf.coef_[i][top], color=color)
    ax.set_title(f"Palabras que más empujan a «{clase}»"); ax.set_xlabel("Peso en el modelo")
plt.tight_layout(); plt.show()

### 👀 A12 · EJEMPLO — Probar frases nuevas con negaciones

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Qué predice el modelo para «La limpieza no es impecable»?
2. ¿Y para «El personal no fue borde»?
3. ¿Por qué falla con las negaciones?

In [ ]:
pruebas = ["La limpieza es impecable", "La limpieza no es impecable", "El personal fue borde", "El personal no fue borde",
           "Vistas increíbles pero un ruido constante"]
for f, p in zip(pruebas, modelo_lr.predict(pruebas)):
    print(f"{p:9s} ← {f}")

### ✏️ A13 · EJERCICIO 4 de 5 — ¿Falla con las negaciones?

**Tu turno:** Observa los resultados y escribe en `falla_negacion` (True/False) si el modelo trata igual las frases "con negación" que las "sin negación" (es decir, ¿se equivoca con "no es impecable"?). Después, en `bigrama_util`, escribe (texto) por qué usar **bigramas** (`ngram_range=(1, 2)`) podría ayudar en una frase. Pista: los bigramas conservan pares como "no es".

**Debe salir:** ✔ Ejercicio 4 correcto (`falla_negacion` = True)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A13).

1. ¿Cuánto vale `falla_negacion` y qué significa?
2. ¿Qué es un bigrama?
3. ¿Por qué ayudan los bigramas con las negaciones?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A13, **después de intentarlo**.

In [ ]:
# pista: compara la predicción de la 1.ª y 2.ª frase; explica que un bigrama como «no fue» mantiene la negación junto a la palabra
falla_negacion = ...
bigrama_util = "..."

assert isinstance(falla_negacion, (bool, np.bool_)), "falla_negacion debe ser True o False."
assert len(bigrama_util) > 30, "Escribe una explicación de al menos una frase en bigrama_util."
print("✔ Ejercicio 4 correcto")

### 👀 A14 · EJEMPLO — Detectar temas por palabras clave y medir su F1

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A14).

1. ¿Qué tema tiene el F1 más bajo y cuánto vale?
2. ¿Qué ventaja tiene el método de palabras clave?
3. ¿Cuál es su límite?

In [ ]:
TEMAS = {
    "limpieza":  ["limpi", "sucia", "sucio", "pelos", "sábanas", "toallas"],
    "ruido":     ["ruid", "silencio", "música", "dormir", "oye", "tranquilo"],
    "desayuno":  ["desayuno", "café", "buffet", "colas"],
    "personal":  ["personal", "recepción", "recepcionista", "trato", "borde", "atendía", "ayudó", "tardaron"],
    "ubicacion": ["ubicación", "situado", "lejos", "centro", "playa", "vistas", "coche", "zona"],
    "precio":    ["precio", "caro", "cuesta", "extras", "cobran", "compensa"],
}

def detectar_temas(texto, diccionario=TEMAS):
    t = limpiar(texto)
    return sorted(tema for tema, claves in diccionario.items() if any(c in t for c in claves))

resenas["temas_detectados"] = resenas["texto"].apply(detectar_temas)
resenas["temas_reales"] = resenas["temas"].fillna("").apply(lambda s: [x for x in s.split(",") if x])

def f1_tema(tema):
    real = resenas["temas_reales"].apply(lambda l: tema in l)
    pred = resenas["temas_detectados"].apply(lambda l: tema in l)
    return f1_score(real, pred)

f1_por_tema = pd.Series({t: f1_tema(t) for t in TEMAS}).sort_values()
print(f1_por_tema.round(2).to_string())

### ✏️ A15 · EJERCICIO 5 de 5 — El tema peor detectado

**Tu turno:** ¿Cuál es el tema con peor F1 (`tema_peor`, texto) y cuántas reseñas del corpus mencionan **realmente** el tema `precio` (`n_precio`, entero, usando `temas_reales`)? Después mira los casos donde falla ese tema y decide qué cambio harías en el diccionario (¿quitar una clave demasiado amplia? ¿añadir otra?).

**Debe salir:** `Tema peor: ruido | n precio: 455` y ✔ Ejercicio 5 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A15).

1. ¿Qué valen `tema_peor` y `n_precio`?
2. ¿Qué diferencia hay entre `max()` e `idxmin()` aquí?
3. ¿Para qué sirve contar `n_precio`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A15, **después de intentarlo**.

In [ ]:
# pista: idxmin de f1_por_tema; contar cuántas listas de temas_reales contienen "precio"
tema_peor = ...
n_precio = ...

assert tema_peor in TEMAS, "tema_peor debe ser una de las claves de TEMAS."
assert 300 < n_precio < 700, "n_precio: cuenta las reseñas cuya lista temas_reales contiene «precio»."
print("Tema peor:", tema_peor, "| n precio:", n_precio)
print("✔ Ejercicio 5 correcto")

### 👀 A16 · EJEMPLO — Dónde falla el tema peor

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A16).

1. ¿Cuántas reseñas fallan con el tema ruido?
2. ¿Por qué las detecta como «ruido» si no lo son?
3. ¿Qué cambio harías en el diccionario?

In [ ]:
errores = resenas[resenas["temas_reales"].apply(lambda l: tema_peor in l) != resenas["temas_detectados"].apply(lambda l: tema_peor in l)]
print(f"{len(errores)} reseñas donde falla «{tema_peor}». Ejemplos:")
for _, fila in errores.head(5).iterrows():
    print("-", fila["texto"], "| real:", fila["temas_reales"], "| detectado:", fila["temas_detectados"])

### 👀 A17 · EJEMPLO — Los n-gramas más frecuentes en las reseñas negativas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A17).

1. ¿Cuál es la expresión más frecuente y cuántas veces aparece?
2. ¿Por qué aparecen tantas expresiones con la misma frecuencia?
3. ¿Qué muestran los n-gramas que no muestra una sola palabra?

In [ ]:
# Palabras de "relleno" que abren muchas reseñas y no dicen nada de la estancia
RELLENO = PALABRAS_VACIAS | {"pasamos", "semana", "aquí", "escapada", "fin", "viajamos", "familia", "fuimos", "trabajo", "general", "todo"}

def sin_relleno(texto):
    return " ".join(t for t in limpiar(texto).split() if t not in RELLENO)

def top_ngramas(textos, n=(2, 3), k=8):
    cv = CountVectorizer(preprocessor=sin_relleno, ngram_range=n, min_df=5)
    M = cv.fit_transform(textos)
    frec = pd.Series(np.asarray(M.sum(axis=0)).ravel(), index=cv.get_feature_names_out())
    return frec.sort_values(ascending=False).head(k)

neg = resenas.loc[resenas["sentimiento"] == "negativo", "texto"]
print(top_ngramas(neg).to_string())

### ✏️ A18 · MINI-RETO — Panel de quejas por tema

**Tu turno:** Construye `quejas`, una serie con el **número de reseñas negativas por tema** (usa `temas_detectados` y solo reseñas con `sentimiento == "negativo"`), ordenada de mayor a menor. Dibuja un gráfico de barras y guarda en `queja_principal` el tema más frecuente. **Antes de fiarte del resultado:** `temas_detectados` sale del diccionario de palabras clave, no de las etiquetas del corpus. Compara tu `quejas` con `resenas.loc[resenas["sentimiento"] == "negativo", "temas_reales"].explode().value_counts()`: si el tema principal no coincide, la lista de palabras de ese tema es demasiado amplia (recuerda el ejercicio 5).

**Debe salir:** un gráfico de barras de las quejas por tema (`ruido` 201 · `limpieza` 171 · `precio` 165 · `desayuno` 162 · `ubicacion` 160 · `personal` 147) y ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A18).

1. ¿Qué tema sale como queja principal con las palabras clave y cuántas reseñas tiene?
2. ¿Coincide con el tema principal según las etiquetas reales?
3. ¿Qué le dirías a la dirección antes de enseñarle el panel?
4. ¿Qué pasaría si el cliente tuviera reseñas reales con ironía?

In [ ]:
# pista: explode() sobre temas_detectados de las negativas, value_counts()
quejas = ...
queja_principal = ...

fig, ax = plt.subplots()
quejas.sort_values().plot.barh(ax=ax, color="#c0463a")
ax.set_title("Reseñas negativas por tema"); ax.set_xlabel("Número de reseñas")
plt.show()
assert quejas.is_monotonic_decreasing, "quejas debe estar ordenada de mayor a menor."
assert queja_principal in TEMAS, "queja_principal debe ser un tema del diccionario."
assert quejas.sum() >= 300, "Cuenta solo reseñas negativas, un tema por fila (usa explode)."
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `re` (expresiones regulares), `numpy`, `pandas`, `matplotlib` y, de `scikit-learn`, los vectorizadores de texto (`CountVectorizer`, `TfidfVectorizer`), la separación de datos, la regresión logística, Naive Bayes y las métricas.
    - `warnings.filterwarnings("ignore")` oculta avisos que no afectan a los resultados, `np.random.seed(SEMILLA)` fija la semilla y `plt.rcParams.update(...)` da un estilo común a los gráficos.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 're' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Conocer los datos

#### A2 · Ejemplo — Cargar las reseñas

**Contexto.** TurisData recibe 1 500 reseñas. La dirección no puede leerlas todas: quiere saber *qué opina la gente* (sentimiento) y *de qué habla* (temas).

Cada reseña tiene texto, nota (1-5), sentimiento (positivo / neutro / negativo) y temas mencionados.
**Aviso:** son reseñas *sintéticas* (inventadas para el curso): más repetitivas y fáciles que las reales.

- **Qué hace:**
    - `pd.read_csv(dato("resenas.csv"))` lee las reseñas; `resenas.shape` da su forma y `resenas.head()` muestra las 5 primeras filas.
- **Qué debe salir:**
    - `(1500, 5)` y una tabla con las columnas `id_resena`, `texto`, `nota`, `sentimiento` y `temas` (por ejemplo, `ruido,ubicacion`).
- **Cómo leerlo:**
    - Cada reseña tiene su texto, una nota de 1 a 5, el sentimiento (positivo, neutro o negativo) y los temas que menciona. **Aviso:** son reseñas *sintéticas* (inventadas para el curso): más repetitivas y fáciles que las reales.
- **Si falla:**
    - `NameError: name 'dato' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas reseñas y columnas hay?* → **1 500** reseñas y **5** columnas.
2. *¿Qué información tiene cada reseña?* → El **texto**, la **nota** (1–5), el **sentimiento** y los **temas** mencionados.
3. *¿Por qué hay que tener cuidado con estos datos?* → Porque son **sintéticos**: más repetitivos y fáciles que las reseñas reales.

</details>

#### A3 · Ejemplo — Distribución del sentimiento y reseñas de ejemplo

- **Qué hace:**
    - `value_counts(normalize=True)` da la proporción de cada sentimiento.
    - `resenas["longitud"]` es el número de palabras de cada reseña (`str.split().str.len()`) y se imprime su media.
    - El bucle imprime 3 reseñas al azar (`sample(3, random_state=SEMILLA)`).
- **Qué debe salir:**
    - `positivo 0.469`, `negativo 0.319`, `neutro 0.212`; `Palabras por reseña (media): 14.0` y tres reseñas de ejemplo.
- **Cómo leerlo:**
    - Casi la mitad son positivas (46,9 %), un tercio negativas y un 21 % neutras. Son textos cortos (14 palabras de media), hechos de frases sueltas muy parecidas entre sí.
- **Si falla:**
    - `NameError: name 'resenas' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué sentimiento es el más frecuente y qué proporción tiene?* → **Positivo**, con el **46,9 %**.
2. *¿Cuántas palabras tiene una reseña de media?* → **14**.
3. *¿Qué proporción acertaría un modelo que dijera siempre «positivo»?* → El **46,9 %**: es la referencia de «no saber nada».

</details>

## 2. Limpieza y tokenización

#### A4 · Ejercicio 1 — Limpiar un texto

**Idea.** El ordenador ve "Bueno", "bueno!" y "BUENO" como palabras distintas. La limpieza las unifica:
minúsculas, quitar signos y separar en **tokens** (palabras). Luego se quitan las **palabras vacías**
(*stopwords*: "de", "el", "la"...), que casi no aportan. Ojo: **no quitamos "no"**, porque cambia el sentido.

- **Qué hace:**
    - `texto.lower()` pasa a minúsculas.
    - `re.sub(r"[^a-záéíóúüñ\s]", " ", texto)` cambia por un espacio todo lo que **no** sea una letra (incluidas tildes, ü y ñ) o un espacio: signos, números y símbolos.
    - `" ".join(texto.split())` colapsa los espacios repetidos en uno.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Cómo leerlo:**
    - `limpiar("¡Qué BUENO!...  Sí")` da `"qué bueno sí"`: se conservan las tildes, que forman parte de las palabras en español.
- **Si falla:**
    - `Revisa limpiar: minúsculas, signos fuera y espacios únicos.` → falta alguno de los tres pasos (minúsculas, quitar signos o colapsar espacios).
    - `Los números y símbolos también se eliminan.` → la expresión regular deja pasar los números; solo debe conservar letras y espacios.
    - `NameError: name 're' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué devuelve `limpiar("Precio 3€, ok")`?* → **"precio ok"**: los números y símbolos desaparecen.
2. *¿Por qué la expresión regular incluye `áéíóúüñ`?* → Para **conservar las tildes y la ñ**: son letras del español y cambian la palabra.
3. *¿Para qué sirve `" ".join(texto.split())`?* → Para **colapsar los espacios repetidos** en uno solo.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def limpiar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    return " ".join(texto.split())
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A5 · Ejemplo — Quitar las palabras vacías y tokenizar

- **Qué hace:**
    - `PALABRAS_VACIAS` es un conjunto de palabras que casi no aportan («de», «el», «la»…). **No incluye «no»**, porque cambia el sentido.
    - `tokenizar(texto)` limpia el texto, lo separa en palabras (*tokens*) y, por defecto, quita las vacías.
    - Se prueba con una frase y se crea la columna `limpio` con todas las reseñas limpias.
- **Qué debe salir:**
    - `['limpieza', 'habitación', 'impecable']`.
- **Cómo leerlo:**
    - De «La limpieza de la habitación era impecable!» quedan solo las tres palabras con significado.
- **Si falla:**
    - `NameError: name 'limpiar' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tokens salen de la frase de ejemplo?* → **limpieza**, **habitación** e **impecable**.
2. *¿Por qué no se quita «no» de las palabras vacías?* → Porque **cambia el sentido** de la frase («no es bueno»).
3. *¿Qué es un token?* → Una **palabra** (o unidad de texto) obtenida al separar el texto.

</details>

#### A6 · Ejercicio 2 — Las 5 palabras más frecuentes

- **Qué hace:**
    - `Counter(t for texto in resenas["texto"] for t in tokenizar(texto))` cuenta cuántas veces aparece cada token en **todo** el corpus (`tokenizar` ya quita las palabras vacías).
    - `contador.most_common(5)` devuelve los 5 más frecuentes como lista de pares `(palabra, veces)` (`top5`).
- **Qué debe salir:**
    - `[('semana', 504), ('desayuno', 391), ('todo', 382), ('habitación', 284), ('escapada', 256)]` y `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Las palabras más frecuentes no son las más informativas: «semana», «todo» y «escapada» salen de las frases con que empiezan muchas reseñas («Pasamos una semana aquí», «Escapada de fin de semana»). Solo «desayuno» y «habitación» hablan de la estancia.
- **Si falla:**
    - `top5 debe ser una lista de 5 pares (palabra, veces)` → no has usado `most_common(5)`.
    - `Quita las palabras vacías con tokenizar()` → has contado las palabras sin quitar las vacías (usa `tokenizar`).
    - `NameError: name 'tokenizar' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuál es la palabra más frecuente y cuántas veces aparece?* → **semana**, **504** veces.
2. *¿Por qué «semana», «todo» y «escapada» salen tan arriba?* → Porque aparecen en **frases de relleno** con que empiezan muchas reseñas; frecuente no significa informativo.
3. *¿Qué devuelve `most_common(5)`?* → Una lista de **5 pares** `(palabra, veces)` de mayor a menor.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
from collections import Counter
contador = Counter(t for texto in resenas["texto"] for t in tokenizar(texto))
top5 = contador.most_common(5)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. De texto a números: bolsa de palabras y TF-IDF

#### A7 · Ejemplo — Bolsa de palabras y TF-IDF con tres frases

**Bolsa de palabras (BoW):** una columna por palabra del vocabulario; cada reseña cuenta cuántas veces aparece.
Se pierde el orden, pero es muy útil.

**TF-IDF:** igual, pero las palabras que aparecen en *todas* las reseñas pesan menos y las **distintivas** pesan más
(*term frequency × inverse document frequency*).

Observa que "desayuno" (aparece en dos de tres) pesa menos que "sucia" (solo en una) en TF-IDF.

- **Qué hace:**
    - `CountVectorizer()` crea la **bolsa de palabras**: una columna por palabra del vocabulario y, por fila, cuántas veces aparece en cada frase.
    - `TfidfVectorizer()` hace lo mismo pero con **pesos TF-IDF**: las palabras que aparecen en muchas frases pesan menos.
    - Cada resultado se convierte en una tabla (`pd.DataFrame`) con las palabras como columnas.
- **Qué debe salir:**
    - Dos tablas de 3 filas (`r1`, `r2`, `r3`) y 10 columnas (`desayuno`, `el`, `era`, `escaso`, `estaba`, `habitación`, `la`, `merece`, `pena`, `sucia`): la primera con cuentas (0 y 1) y la segunda con pesos decimales.
- **Cómo leerlo:**
    - En TF-IDF, «desayuno» (en dos de las tres frases) pesa **0,43** y **0,39**, menos que «sucia» (solo en una), que pesa **0,53**: las palabras distintivas pesan más.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué contiene cada columna de la bolsa de palabras?* → El **número de veces** que aparece una palabra en cada frase.
2. *¿Qué peso tiene «sucia» en TF-IDF y por qué es mayor que el de «desayuno»?* → **0,53** frente a 0,43 y 0,39: porque «sucia» aparece en **una** sola frase (es más distintiva).
3. *¿Qué se pierde en la bolsa de palabras?* → **El orden** de las palabras.

</details>

## 4. Clasificar el sentimiento

#### A8 · Ejemplo — Partir en entrenamiento y prueba

Partimos en entrenamiento y prueba (estratificado para mantener las proporciones) y **ajustamos el vectorizador solo con el entrenamiento**
(misma idea que el escalado sin fuga). Usamos `Pipeline` para no equivocarnos.

- **Qué hace:**
    - `train_test_split(resenas["texto"], resenas["sentimiento"], test_size=0.25, stratify=..., random_state=SEMILLA)` separa un 25 % para probar, manteniendo la proporción de cada sentimiento (`stratify`).
    - Se importa `make_pipeline`, que encadenará el vectorizador y el clasificador para que el vectorizador **se ajuste solo con el entrenamiento**.
- **Qué debe salir:**
    - `1125 para entrenar, 375 para probar`.
- **Cómo leerlo:**
    - Se trabaja con el texto en bruto: el vectorizador se incluirá dentro del *pipeline* para no filtrar información de la prueba.
- **Si falla:**
    - `NameError: name 'resenas' is not defined` → falta ejecutar los ejemplos anteriores.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas reseñas hay para entrenar y para probar?* → **1 125** y **375**.
2. *¿Para qué sirve `stratify`?* → Para que entrenamiento y prueba tengan **la misma proporción** de cada sentimiento.
3. *¿Por qué se usa un *pipeline*?* → Para que el vectorizador se ajuste **solo con el entrenamiento** (sin fuga) y no equivocarnos al aplicarlo.

</details>

#### A9 · Ejercicio 3 — Regresión logística frente a Naive Bayes

**Ejercicio 3 · 🏫 en clase.** Crea y entrena dos modelos y guarda su exactitud en prueba:
- `modelo_lr`: `TfidfVectorizer(preprocessor=limpiar)` + `LogisticRegression(max_iter=1000)` → `acc_lr`
- `modelo_nb`: `CountVectorizer(preprocessor=limpiar)` + `MultinomialNB()` → `acc_nb`

Usa `make_pipeline(vectorizador, clasificador)` y `.fit(X_ent, y_ent)`.

- **Qué hace:**
    - `make_pipeline(TfidfVectorizer(preprocessor=limpiar), LogisticRegression(max_iter=1000)).fit(X_ent, y_ent)` crea y entrena el modelo de regresión logística con TF-IDF (`modelo_lr`); usa tu `limpiar` como preprocesado.
    - `make_pipeline(CountVectorizer(preprocessor=limpiar), MultinomialNB()).fit(X_ent, y_ent)` crea y entrena Naive Bayes con bolsa de palabras (`modelo_nb`).
    - `accuracy_score(y_test, modelo.predict(X_test))` da la exactitud de cada uno en prueba. La comprobación calcula también la exactitud de un clasificador que siempre dice la clase mayoritaria.
- **Qué debe salir:**
    - `Regresión logística + TF-IDF : 0.992`, `Naive Bayes + bolsa de palabras: 0.989`, `(Un clasificador que siempre dice la clase mayoritaria acertaría 0.469)` y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - Los dos aciertan casi todo (99 %) frente al 46,9 % de la referencia. Una exactitud tan alta es sospechosa en la vida real: aquí es posible porque las reseñas **sintéticas** son repetitivas y fáciles.
- **Si falla:**
    - `Las dos exactitudes deberían superar 0.7. Revisa el pipeline y que uses X_ent/y_ent para entrenar.` → falta el `fit` con `X_ent`/`y_ent`, o has entrenado con la prueba.
    - `NameError: name 'limpiar' is not defined` → falta el ejercicio 1.
    - `NameError: name 'X_ent' is not defined` → falta el ejemplo de la partición.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud logran los dos modelos y cuál es la referencia?* → **0,992** (regresión logística) y **0,989** (Naive Bayes), frente a **0,469** de predecir siempre la clase mayoritaria.
2. *¿Qué hace `preprocessor=limpiar` en el vectorizador?* → Aplica **tu función de limpieza** a cada texto antes de contar las palabras.
3. *¿Por qué no hay que fiarse tanto de un 99 %?* → Porque las reseñas son **sintéticas, repetitivas y fáciles**: con reseñas reales saldría bastante menos.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
modelo_lr = make_pipeline(TfidfVectorizer(preprocessor=limpiar), LogisticRegression(max_iter=1000)).fit(X_ent, y_ent)
modelo_nb = make_pipeline(CountVectorizer(preprocessor=limpiar), MultinomialNB()).fit(X_ent, y_ent)
acc_lr = accuracy_score(y_test, modelo_lr.predict(X_test))
acc_nb = accuracy_score(y_test, modelo_nb.predict(X_test))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A10 · Ejemplo — Informe de clasificación y matriz de confusión

- **Qué hace:**
    - `classification_report(y_test, modelo_lr.predict(X_test))` muestra la precisión, el *recall* y el F1 de cada sentimiento.
    - `ConfusionMatrixDisplay.from_predictions(...)` dibuja la matriz de confusión (filas = real, columnas = predicho).
- **Qué debe salir:**
    - Una tabla con `negativo` (precisión 0,98, recall 1,00; 120 reseñas), `neutro` (1,00 y 0,96; 79) y `positivo` (1,00 y 1,00; 176), con exactitud 0,99, y una matriz de confusión casi diagonal.
- **Cómo leerlo:**
    - Se confunden muy pocas: el *recall* del neutro es 0,96 (3 de 79 neutras se predicen mal) y alguna neutra se clasifica como negativa.
- **Si falla:**
    - `NameError: name 'modelo_lr' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué clase tiene el *recall* más bajo y cuánto vale?* → **neutro**, con **0,96**.
2. *¿Qué indican los números fuera de la diagonal de la matriz?* → **Errores**: reseñas clasificadas con un sentimiento distinto del real.
3. *¿Cuántas reseñas de prueba hay de cada sentimiento?* → **120** negativas, **79** neutras y **176** positivas.

</details>

#### A11 · Ejemplo — ¿Qué palabras pesan en el modelo?

La gran ventaja de los modelos clásicos es que se pueden **explicar**: cada palabra tiene un peso hacia cada clase.

- **Qué hace:**
    - `modelo_lr.named_steps[...]` saca el vectorizador y el clasificador del *pipeline*; `vocab` es el vocabulario.
    - Para las clases `positivo` y `negativo`, `clf.coef_[i]` son los pesos de cada palabra hacia esa clase y `np.argsort(...)[-8:]` selecciona las 8 con más peso; se dibujan en barras.
- **Qué debe salir:**
    - Dos gráficos de barras horizontales: las 8 palabras que más empujan a «positivo» y a «negativo».
- **Cómo leerlo:**
    - Pesan lo que se espera: **impecable**, **limpio** y **amabilísimo** empujan a positivo; **sucia**, **borde**, **caro** y **demasiado** a negativo. También aparecen palabras sin carga propia («las», «del») y **«no»** entre las negativas: eso se puede revisar y es la gran ventaja de los modelos clásicos: **se pueden explicar**.
- **Si falla:**
    - `NameError: name 'modelo_lr' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué palabras empujan más a «negativo»?* → **sucia**, **borde**, **caro**, **demasiado**… (y también «no»).
2. *¿Qué ventaja tiene poder ver los pesos?* → Que el modelo se puede **explicar y revisar**: se ve por qué decide y si se fija en palabras sin sentido.
3. *¿Qué pesos te parecen sospechosos?* → Palabras sin carga propia como **«las»** o **«del»**: señal de que el modelo se apoya en rasgos de las plantillas sintéticas.

</details>

## 5. Los límites: negaciones

#### A12 · Ejemplo — Probar frases nuevas con negaciones

Una bolsa de palabras no entiende el orden. Probemos frases nuevas que el modelo no ha visto.

- **Qué hace:**
    - `pruebas` son 5 frases nuevas, entre ellas pares con y sin negación («es impecable» y «no es impecable»).
    - `modelo_lr.predict(pruebas)` predice el sentimiento de cada una y el bucle imprime cada predicción junto a su frase.
- **Qué debe salir:**
    - `positivo ← La limpieza es impecable`, `positivo ← La limpieza no es impecable`, `negativo ← El personal fue borde`, `negativo ← El personal no fue borde` y `negativo ← Vistas increíbles pero un ruido constante`.
- **Cómo leerlo:**
    - La regresión logística **no entiende la negación**: «no es impecable» sigue siendo positivo y «no fue borde» sigue siendo negativo. La bolsa de palabras ve «impecable» y «borde» y se olvida del «no».
- **Si falla:**
    - `NameError: name 'modelo_lr' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué predice el modelo para «La limpieza no es impecable»?* → **positivo**: no entiende la negación.
2. *¿Y para «El personal no fue borde»?* → **negativo**: también falla, por la misma razón.
3. *¿Por qué falla con las negaciones?* → Porque la bolsa de palabras **pierde el orden** y trata «no» como una palabra suelta.

</details>

#### A13 · Ejercicio 4 — ¿Falla con las negaciones?

- **Qué hace:**
    - `modelo_lr.predict([pruebas[1]])[0] == "positivo"` comprueba si el modelo clasifica como positiva la frase **con** negación («La limpieza no es impecable»): `True` si falla.
    - `bigrama_util` es una frase (texto) que explica por qué los bigramas (pares de palabras seguidas, como «no es») conservan parte del orden y de la negación.
- **Qué debe salir:**
    - `✔ Ejercicio 4 correcto`. `falla_negacion` vale `True`.
- **Cómo leerlo:**
    - Con bigramas, el par «no es» (o «no fue») se vuelve una columna propia, y el modelo puede aprender que «no es impecable» no significa lo mismo que «es impecable». Por eso `TfidfVectorizer(ngram_range=(1, 2))` ayuda.
- **Si falla:**
    - `falla_negacion debe ser True o False.` → has guardado otra cosa (por ejemplo, la predicción en lugar de la comparación).
    - `Escribe una explicación de al menos una frase en bigrama_util.` → el texto tiene menos de 30 caracteres.
    - `NameError: name 'pruebas' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale `falla_negacion` y qué significa?* → **`True`**: el modelo trata como positiva la frase con negación («no es impecable»).
2. *¿Qué es un bigrama?* → Una **secuencia de dos palabras seguidas** (por ejemplo, «no es»).
3. *¿Por qué ayudan los bigramas con las negaciones?* → Porque **conservan el «no» junto a la palabra que niega**, así el modelo ve «no es» como una pista propia.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
falla_negacion = modelo_lr.predict([pruebas[1]])[0] == "positivo"
bigrama_util = "Un bigrama junta dos palabras seguidas (p. ej. «no es»), así conserva parte del orden y de la negación."
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Temas por palabras clave

#### A14 · Ejemplo — Detectar temas por palabras clave y medir su F1

**Idea.** Sin entrenar nada: definimos, para cada tema, una lista de palabras que lo delatan. Si la reseña contiene alguna, se asigna el tema.
Es una técnica **transparente y barata**; su límite es que hay que mantener la lista.
Comparamos con la columna `temas` (etiquetas del corpus) usando la puntuación F1 de cada tema.

- **Qué hace:**
    - `TEMAS` es un diccionario: para cada tema (limpieza, ruido, desayuno, personal, ubicación y precio), una lista de raíces de palabras que lo delatan.
    - `detectar_temas(texto)` limpia el texto y devuelve los temas cuyas palabras clave aparecen en él (como parte de alguna palabra).
    - `temas_reales` son las etiquetas del corpus; `f1_tema(tema)` calcula el F1 de la detección de cada tema y `f1_por_tema` los ordena de menor a mayor.
- **Qué debe salir:**
    - `ruido 0.94` y `limpieza`, `desayuno`, `personal`, `ubicacion` y `precio` con `1.00`.
- **Cómo leerlo:**
    - El método transparente y barato funciona casi a la perfección (F1 de 1,00 en cinco de los seis temas) en este corpus sintético. El peor es **ruido** (0,94): su palabra clave `ruid` coincide también con «ruidosa».
- **Si falla:**
    - `NameError: name 'limpiar' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tema tiene el F1 más bajo y cuánto vale?* → **ruido**, con **0,94**.
2. *¿Qué ventaja tiene el método de palabras clave?* → Es **transparente y barato**: no entrena nada y se sabe por qué detecta cada tema.
3. *¿Cuál es su límite?* → Que hay que **mantener la lista** de palabras y una clave demasiado amplia da falsos positivos.

</details>

#### A15 · Ejercicio 5 — El tema peor detectado

- **Qué hace:**
    - `f1_por_tema.idxmin()` da el tema con menor F1 (`tema_peor`).
    - `resenas["temas_reales"].apply(lambda l: "precio" in l).sum()` cuenta cuántas reseñas tienen realmente el tema `precio` en su lista de temas (`n_precio`).
- **Qué debe salir:**
    - `Tema peor: ruido | n precio: 455` y `✔ Ejercicio 5 correcto`.
- **Cómo leerlo:**
    - El tema peor detectado es **ruido**, y 455 de las 1 500 reseñas mencionan realmente el precio.
- **Si falla:**
    - `tema_peor debe ser una de las claves de TEMAS.` → has guardado el valor del F1 en vez del nombre del tema (usa `idxmin()`).
    - `n_precio: cuenta las reseñas cuya lista temas_reales contiene «precio».` → has contado en `temas_detectados` o has contado temas en vez de reseñas.
    - `NameError: name 'f1_por_tema' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valen `tema_peor` y `n_precio`?* → **"ruido"** y **455**.
2. *¿Qué diferencia hay entre `max()` e `idxmin()` aquí?* → `idxmin()` da **el nombre del tema** con el valor mínimo; `min()` solo daría el valor.
3. *¿Para qué sirve contar `n_precio`?* → Para saber **cuántas reseñas hablan realmente** del precio, que sirve de referencia para juzgar el detector.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
tema_peor = f1_por_tema.idxmin()
n_precio = int(resenas["temas_reales"].apply(lambda l: "precio" in l).sum())
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A16 · Ejemplo — Dónde falla el tema peor

- **Qué hace:**
    - `errores` son las reseñas en las que la detección del `tema_peor` no coincide con las etiquetas (detectado y real son distintos).
    - Se imprime cuántas son y 5 ejemplos con su lista real y la detectada.
- **Qué debe salir:**
    - `55 reseñas donde falla «ruido». Ejemplos:` y 5 reseñas, todas con «La zona es ruidosa y poco segura de noche» y `real` sin `ruido` pero `detectado` con él.
- **Cómo leerlo:**
    - El fallo es siempre el mismo: la frase «La zona es ruidosa…» pertenece al tema **ubicación** en las etiquetas, pero la clave `ruid` la marca también como **ruido** (falso positivo). Remedio: **afinar el diccionario**, por ejemplo cambiando la clave `ruid` por otra más precisa.
- **Si falla:**
    - `NameError: name 'tema_peor' is not defined` → falta el ejercicio 5 (o su 🆘 solución).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas reseñas fallan con el tema ruido?* → **55**.
2. *¿Por qué las detecta como «ruido» si no lo son?* → Porque la clave **`ruid` coincide con «ruidosa»** en la frase «La zona es ruidosa…», que las etiquetas asignan a ubicación.
3. *¿Qué cambio harías en el diccionario?* → **Sustituir o acotar la clave `ruid`** (por ejemplo, `ruido` o `se oye`) para que no coincida con «ruidosa».

</details>

## 7. n-gramas: expresiones que se repiten

#### A17 · Ejemplo — Los n-gramas más frecuentes en las reseñas negativas

Un **n-grama** es una secuencia de n palabras seguidas. Los bigramas y trigramas más frecuentes en las reseñas **negativas**
revelan quejas concretas que una sola palabra esconde ("ruido constante").

- **Qué hace:**
    - `RELLENO` son palabras de «relleno» que abren muchas reseñas (por ejemplo, «semana», «escapada»); `sin_relleno(texto)` las quita tras limpiar.
    - `top_ngramas(textos, n=(2, 3), k=8)` cuenta los bigramas y trigramas con `CountVectorizer(ngram_range=...)`, se queda con los que aparecen en al menos 5 reseñas (`min_df=5`) y devuelve los `k` más frecuentes.
    - Se aplica a las reseñas de sentimiento negativo.
- **Qué debe salir:**
    - Una tabla con 8 expresiones y su frecuencia: `constante noche` 69, `ruido constante` 69, `ruido constante noche` 69, `ruidosa poco` 61, `ruidosa poco segura` 61, `poco segura` 61, `poco segura noche` 61 y `zona ruidosa` 61.
- **Cómo leerlo:**
    - Las expresiones revelan quejas concretas que una sola palabra esconde: «ruido constante (de) noche» y «zona ruidosa y poco segura de noche». Aparecen varias veces porque un trigrama contiene bigramas más cortos (los mismos casos se cuentan en cada uno).
- **Si falla:**
    - `NameError: name 'PALABRAS_VACIAS' is not defined` → falta ejecutar el ejemplo de las palabras vacías.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuál es la expresión más frecuente y cuántas veces aparece?* → **"constante noche"**, **"ruido constante"** y **"ruido constante noche"**, todas con **69**.
2. *¿Por qué aparecen tantas expresiones con la misma frecuencia?* → Porque un **trigrama contiene bigramas** más cortos y se cuentan los mismos casos.
3. *¿Qué muestran los n-gramas que no muestra una sola palabra?* → **Quejas concretas** («ruido constante», «zona ruidosa y poco segura») en lugar de palabras sueltas.

</details>

## 8. Mini-reto: panel de quejas

#### A18 · Mini-reto — Panel de quejas por tema

- **Qué hace:**
    - `resenas.loc[resenas["sentimiento"] == "negativo", "temas_detectados"]` toma los temas detectados de las reseñas negativas; `.explode()` pone cada tema en su propia fila y `.value_counts()` cuenta cuántas veces sale cada uno, ya ordenado de mayor a menor (`quejas`).
    - `queja_principal = quejas.index[0]` es el tema más frecuente.
    - El gráfico y las comprobaciones verifican el orden, que la queja principal es un tema del diccionario y que se han contado solo reseñas negativas.
- **Qué debe salir:**
    - Un gráfico de barras horizontales y `✔ Mini-reto correcto`. `quejas`: `ruido` 201, `limpieza` 171, `precio` 165, `desayuno` 162, `ubicacion` 160 y `personal` 147; `queja_principal` vale `"ruido"`.
- **Cómo leerlo:**
    - **Ojo antes de fiarte.** Con las etiquetas reales de esas mismas reseñas, el tema más frecuente es **limpieza** (171) y **ruido** solo 155. El panel sobreestima el ruido (201) por la clave demasiado amplia (`ruid` coincide con «ruidosa»): el diccionario del ejercicio 5 propaga el error al panel.
- **Si falla:**
    - `quejas debe estar ordenada de mayor a menor.` → falta `value_counts()` (que ya ordena) o la has reordenado.
    - `queja_principal debe ser un tema del diccionario.` → no es una clave de `TEMAS` (usa `quejas.index[0]`).
    - `Cuenta solo reseñas negativas, un tema por fila (usa explode).` → no has filtrado por `sentimiento == "negativo"` o no has usado `explode()`.
    - `NameError: name 'temas_detectados'`/`KeyError` → falta el ejemplo de los temas por palabras clave.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tema sale como queja principal con las palabras clave y cuántas reseñas tiene?* → **ruido**, con **201** reseñas negativas.
2. *¿Coincide con el tema principal según las etiquetas reales?* → **No**: con las etiquetas, el primero es **limpieza** (171) y ruido tiene **155**.
3. *¿Qué le dirías a la dirección antes de enseñarle el panel?* → Que el panel **sobreestima el ruido** por una palabra clave demasiado amplia, y que hay que afinar el diccionario y comprobarlo con una muestra revisada a mano.
4. *¿Qué pasaría si el cliente tuviera reseñas reales con ironía?* → Que las palabras clave y la bolsa de palabras **fallarían más** (ironía, negaciones): habría que usar modelos que entiendan el contexto.

</details>

## Preguntas de reflexión
1. La exactitud es muy alta. ¿Por qué no debemos fiarnos tanto de ese número con reseñas **sintéticas** y repetitivas?
2. ¿Qué ventaja tiene un modelo que puedes explicar (pesos de palabras) frente a uno que no?
3. ¿Qué haría falta para manejar ironía ("¡Qué gran idea poner música a las 3 de la mañana!")?
4. Las reseñas pueden contener nombres o datos personales: ¿qué harías antes de guardarlas y analizarlas?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque las reseñas sintéticas están hechas con **frases de plantilla** muy repetitivas: el modelo reconoce las plantillas y no la opinión, así que acierta un 99 %. Con reseñas reales (faltas, ironía, frases largas y variadas) la exactitud sería mucho menor, y no sabemos cuánto hasta probarlo con datos reales.
2. Se puede **explicar** al cliente por qué decide («sucia», «borde», «caro» empujan a negativo), **detectar errores** (por ejemplo, palabras sin sentido con mucho peso) y cumplir con el derecho a una explicación (RGPD). Un modelo que no se explica es una caja negra difícil de revisar y de defender.
3. Haría falta entender el **contexto y el orden**, no solo las palabras: modelos de lenguaje (como los *transformers*), datos etiquetados con ironía y, para el tono, quizás emojis o contexto de la conversación. Los bigramas ayudan con la negación pero no resuelven la ironía.
4. **Anonimizar** (quitar o sustituir nombres, correos, teléfonos), guardar solo lo necesario, restringir quién accede y tener una **base legal** y un plazo de conservación (RGPD). Es mejor anonimizar antes de guardar que después.

</details>

## Resumen: qué has aprendido
- Limpiar, tokenizar y quitar palabras vacías (sin quitar "no").
- Bolsa de palabras y TF-IDF convierten texto en números; se ajustan **solo con el entrenamiento**.
- Regresión logística y Naive Bayes clasifican bien y se pueden explicar; fallan con negaciones.
- Las palabras clave dan temas de forma transparente; los n-gramas muestran quejas concretas.